# 06 — Dal dominio 0.5–6 MeV al dominio 10–20 MeV

Fino alla v0.18 il toy lavorava a 0.5–6 MeV. Dalla v0.19 il dominio è 10–20 MeV, quello di RIPTIDE.
Il cambio non è un semplice riscalamento: alcune approssimazioni che valevano a bassa energia
vanno ricontrollate. Questo notebook raccoglie le modifiche e il motivo di ciascuna.

1. **Cinematica relativistica esatta**: a 20 MeV lo scarto dalla formula classica è mezzo
   $\sigma_{E_p}$ ed è comune a tutti gli eventi, quindi produrrebbe un plateau.
2. **Isotropia in CM contro ENDF/B-VIII.0**: la distribuzione angolare n–p vera è anisotropa
   di qualche per cento. Si verifica che non cambi la ricostruzione.
3. **Prior fisico $\sin 2\theta_p$ nel Caso A**: con eventi generati dalla cinematica in CM, il
   prior piatto su $\theta_p$ biasava $E_n$ combinato.
4. **Ottimizzazione di `loglik` del Caso A**: un solo buffer riusato in place; 1 evento sotto 1 ms.
5. **Punto aperto del Caso C**: la calibrazione di $\mu_E$ e $\log\sigma_E$ non è raggiunta;
   diagnosi e possibili correzioni, ancora da decidere.

Etichette: **(a)** fatto consolidato · **(b)** stima toy con assunzioni esplicite · **(c)**
verificato con Monte Carlo · **(d)** ipotesi da testare.

I calcoli qui sono leggeri (pochi secondi). I risultati dei run pesanti si leggono da `outputs/`.

In [ ]:
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre
from scipy.stats import norm

from riptide_toy import combine, forward_model, grids, kinematics, posterior_A, priors
from riptide_toy.constants import (EN_MAX, EN_MIN, NP_LEGENDRE_COEFFS, NP_LEGENDRE_ENERGIES,
                                   PROTON_MASS_MEV, SEED, SIGMA_EP, SIGMA_THETA)

sys.path.insert(0, "../scripts")
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(SEED)
OUT = Path("..") / "outputs"


def classical_proton_energy(En: np.ndarray, theta_p: np.ndarray) -> np.ndarray:
    """Energia del protone nel limite non relativistico, Ep = En cos^2(theta_p).

    Args:
        En: energia del neutrone, MeV, forma qualsiasi.
        theta_p: angolo di rinculo in lab, rad, forma compatibile.

    Ritorna:
        Ep in MeV, forma del broadcasting.
    """
    return En * np.cos(theta_p) ** 2


def cm_events(rng: np.random.Generator, En: float, n: int,
              anisotropic: bool = False) -> tuple[np.ndarray, np.ndarray]:
    """Eventi veri (theta_p, Ep) a energia fissa, dalla cinematica relativistica in CM.

    Args:
        rng: generatore numpy.
        En: energia del neutrone, MeV.
        n: numero di eventi.
        anisotropic: True = distribuzione angolare ENDF/B-VIII.0, False = isotropa.

    Ritorna:
        (theta_p in rad, Ep in MeV), ciascuno forma (n,).
    """
    En_arr = np.full(n, En)
    if anisotropic:
        theta_cm = kinematics.sample_cm_angle_anisotropic(
            rng, En_arr, np.asarray(NP_LEGENDRE_ENERGIES), np.asarray(NP_LEGENDRE_COEFFS))
    else:
        theta_cm = kinematics.sample_cm_angle(rng, n)
    theta_p = kinematics.recoil_angle_from_cm(theta_cm, En_arr)
    return theta_p, kinematics.proton_energy(En_arr, theta_p)


def posterior_summary(log_post: np.ndarray, grid: np.ndarray) -> tuple[float, float]:
    """Media e deviazione standard di un posterior a griglia.

    Args:
        log_post: log-posterior non normalizzato, forma (n,).
        grid: griglia, forma (n,), stessa unità del risultato.

    Ritorna:
        (media, std), unità della griglia.
    """
    w = np.exp(log_post - log_post.max())
    w /= w.sum()
    mean = float(np.sum(w * grid))
    return mean, float(np.sqrt(np.sum(w * (grid - mean) ** 2)))

## 1. Cinematica relativistica

Per lo scattering elastico n–p a masse uguali $m$, con bersaglio fermo, la conservazione del
quadrimpulso dà (a):

$$E_p = \frac{2m\,E_n\cos^2\theta_p}{2m + E_n\sin^2\theta_p}, \qquad \theta_p \le 90^\circ .$$

Per $E_n \ll 2m$ si ritrova la formula classica $E_p = E_n\cos^2\theta_p$. Il rapporto fra le due è
$2m/(2m + E_n\sin^2\theta_p)$, quindi lo scarto relativo vale circa $E_n\sin^2\theta_p/(2m)$.
Con $2m = 1876.5$ MeV:

* a 3 MeV e 45° lo scarto è $3 \cdot 0.5/1876.5 \approx 0.08\%$: trascurabile, come nella v0.18;
* a 20 MeV e 45° è $20 \cdot 0.5/1876.5 \approx 0.53\%$, cioè circa 0.05 MeV su $E_p \approx 10$ MeV.

0.05 MeV è mezzo $\sigma_{E_p}$ (0.10 MeV). Da solo non sembra molto. Però non è rumore: ha lo
stesso segno per tutti gli eventi, quindi non si media via combinando $N$ eventi.

In [ ]:
theta = np.deg2rad(45.0)
print(" E_n [MeV]   Ep esatta   Ep classica   scarto [MeV]   scarto / sigma_Ep")
for En in (3.0, 10.0, 15.0, 20.0):
    exact = kinematics.proton_energy(En, theta)
    classic = classical_proton_energy(En, theta)
    print(f"   {En:5.1f}     {exact:8.4f}    {classic:8.4f}      {classic - exact:7.4f}        {(classic - exact) / SIGMA_EP:5.2f}")

theta_deg = np.linspace(0.0, 90.0, 181)
fig, ax = plt.subplots(figsize=(6, 3.5))
for En in (10.0, 15.0, 20.0):
    gap = classical_proton_energy(En, np.deg2rad(theta_deg)) - kinematics.proton_energy(En, np.deg2rad(theta_deg))
    ax.plot(theta_deg, gap / SIGMA_EP, label=f"$E_n$ = {En:.0f} MeV")
ax.set_xlabel(r"$\theta_p$ [°]")
ax.set_ylabel(r"(classica − esatta) / $\sigma_{E_p}$")
ax.legend()
ax.set_title("Scarto della formula classica")
plt.tight_layout()
plt.show()

**Lettura.** Lo scarto è sempre positivo: la formula classica sovrastima $E_p$. Il massimo è a 45°,
dove $\cos^2\theta\sin^2\theta$ è massimo, e cresce linearmente con $E_n$ (a).

### Perché uno scarto comune produce un plateau

Si supponga di ricostruire con la formula classica eventi generati con quella esatta, senza smearing.
La stima di $E_n$ da un evento è $\hat E_n = E_p/\cos^2\theta_p = E_n\,2m/(2m + E_n\sin^2\theta_p)$.
Con scattering isotropo in CM $\cos^2\theta_p$ è uniforme, quindi $\langle\sin^2\theta_p\rangle = 1/2$, e
al primo ordine

$$\langle \hat E_n - E_n \rangle \approx -\frac{E_n^2}{4m}.$$

A 15 MeV: $-225/3753 \approx -0.060$ MeV (b). L'incertezza statistica del Caso A scende come
$\sigma_1/\sqrt N$, con $\sigma_1 \approx 1.3$ MeV per evento a 15 MeV, dominata da $\sigma_\theta$.
Il bias diventa grande quanto l'incertezza per $N^* \approx (1.3/0.060)^2 \approx 470$. Oltre $N^*$
il posterior contrae attorno al valore sbagliato: è il plateau visto nel notebook 02 con lo shift
del 2%, solo più piccolo (b).

La cella sotto verifica la stima del bias con un Monte Carlo sugli eventi veri (c). Verifica anche
un fatto che rende comoda la generazione (a): con scattering isotropo in CM l'energia trasferita è
$E_p = E_n(1-\cos\theta_\mathrm{CM})/2$ anche relativisticamente, quindi $E_p \sim U(0, E_n)$
esattamente, con media $E_n/2$ e varianza $E_n^2/12$. Cambia solo l'angolo $\theta_p$
associato a ogni $E_p$.

In [ ]:
En_true = 15.0
theta_p, Ep = cm_events(rng, En_true, 200_000)
En_classic = Ep / np.cos(theta_p) ** 2
bias_mc = En_classic.mean() - En_true
bias_err = En_classic.std() / np.sqrt(En_classic.size)
bias_th = -En_true ** 2 / (4 * PROTON_MASS_MEV)
print(f"bias dell'inversione classica a {En_true} MeV: MC {bias_mc:+.4f} +- {bias_err:.4f} MeV,"
      f" primo ordine {bias_th:+.4f} MeV")
print(f"N* = (1.3 / |bias|)^2 ~ {(1.3 / abs(bias_mc)) ** 2:.0f}")

for En in (10.0, 20.0):
    _, Ep_check = cm_events(rng, En, 200_000)
    print(f"E_n = {En:4.1f}: <Ep> = {Ep_check.mean():.3f} (atteso {En / 2:.3f}),"
          f" Var = {Ep_check.var():.2f} (atteso {En ** 2 / 12:.2f})")

**Lettura.** Il Monte Carlo conferma la stima al primo ordine: bias di circa −0.06 MeV a 15 MeV e
$N^*$ di qualche centinaio (c). Le checklist arrivano a $N = 1000$, quindi la formula classica non
è accettabile. Dalla v0.19 `kinematics.proton_energy` è esatta e `forward_model` la usa ovunque.
Nei termini che dipendono da $E_n$ entra anche il fattore relativistico della densità delle tracce,
$4m\,c\,(2m+E_n)/[2\pi(2m+E_n\sin^2\theta_p)^2]$, che si scosta da $c/\pi$ al più dell'1% a 20 MeV (a).

Restano due approssimazioni non relativistiche, entrambe dichiarate (b):

* il prior su $\theta_p$ del Caso A (sezione 3);
* `track_angle_cdf`, con uno scarto ≤ 0.003 sulla funzione di ripartizione a 20 MeV.

La checklist del Caso A a 10–20 MeV non mostra plateau (notebook 02).

## 2. Isotropia in CM contro ENDF/B-VIII.0

A bassa energia lo scattering n–p è isotropo in CM. A 10–20 MeV non lo è del tutto. La
valutazione ENDF/B-VIII.0 dell'idrogeno-1 (MAT 125, MF4/MT2) dà la distribuzione angolare del
neutrone in CM come serie di Legendre (a):

$$f(\mu) = \sum_l \frac{2l+1}{2}\,a_l(E_n)\,P_l(\mu), \qquad a_0 = 1, \quad \mu = \cos\theta_\mathrm{CM}.$$

I coefficienti $a_1,\dots,a_6$ da 10 a 20 MeV, a passi di 0.5 MeV, sono in
`constants.NP_LEGENDRE_COEFFS`. Un neutrone diffuso all'indietro ($\mu = -1$) cede tutta l'energia
al protone. Un eccesso all'indietro vuol dire quindi più protoni di alta energia, e lo spettro di
$E_p$ non è più esattamente piatto.

In [ ]:
energies = np.asarray(NP_LEGENDRE_ENERGIES)
coeffs = np.asarray(NP_LEGENDRE_COEFFS)
mu = np.linspace(-1.0, 1.0, 401)


def legendre_density(mu: np.ndarray, a: np.ndarray) -> np.ndarray:
    """Densita' angolare in CM da coefficienti di Legendre (a_1, ..., a_L), a_0 = 1.

    Args:
        mu: cos(theta_CM), forma (n,).
        a: coefficienti (a_1, ..., a_L), forma (L,).

    Ritorna:
        f(mu), densita' su [-1, 1] (integrale 1), forma (n,).
    """
    full = np.concatenate([[1.0], a])
    return legendre.legval(mu, (2 * np.arange(full.size) + 1) / 2 * full)


fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for En in (10.0, 14.0, 20.0):
    i = int(np.argmin(np.abs(energies - En)))
    f = legendre_density(mu, coeffs[i])
    ratio = legendre_density(np.array([-1.0]), coeffs[i])[0] / legendre_density(np.array([0.0]), coeffs[i])[0]
    axes[0].plot(mu, f / 0.5, label=f"{En:.0f} MeV, f(−1)/f(0) = {ratio:.3f}")
axes[0].axhline(1.0, color="k", lw=0.8, ls="--")
axes[0].set_xlabel(r"$\mu = \cos\theta_\mathrm{CM}$")
axes[0].set_ylabel("f / isotropa")
axes[0].legend(fontsize=8)
axes[0].set_title("Distribuzione angolare n–p (ENDF/B-VIII.0)")

En_spec = 20.0
for aniso, label in ((False, "isotropa"), (True, "ENDF/B-VIII.0")):
    _, Ep_spec = cm_events(rng, En_spec, 400_000, anisotropic=aniso)
    axes[1].hist(Ep_spec / En_spec, bins=40, range=(0, 1), density=True, histtype="step", label=label)
axes[1].set_xlabel(r"$E_p / E_n$")
axes[1].set_ylabel("densità")
axes[1].set_title(f"Spettro del protone a {En_spec:.0f} MeV")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lettura.** L'anisotropia c'è e cresce con l'energia: $f(-1)/f(0) = 1.034$ a 10 MeV, 1.061 a
14 MeV, 1.117 a 20 MeV (a, dalla valutazione). Lo spettro di $E_p$ a 20 MeV pende di qualche per
cento verso le energie alte. Per simmetria azimutale l'anisotropia non può spostare $\Omega_n$.
Può spostare le stime di energia ($E_n$ nel Caso A, $\mu_E$ nel Caso C), perché cambia il legame
fra $E_p$ e $E_n$ in media (d, prima del test).

### Stress test

`scripts/stress_anisotropia.py` genera con la distribuzione ENDF e ricostruisce con il modello
isotropo. Gli esperimenti sono appaiati: stesso seme e stesso flusso di numeri casuali con le due
generazioni, quindi la differenza isola l'effetto dell'anisotropia. Il criterio è stato fissato
prima del run: l'anisotropia è trascurabile se il bias aggiunto resta sotto 0.2 std posteriori al
$N$ massimo e la coverage resta nei limiti binomiali.

Il file salvato in `outputs/` contiene oggi solo il Caso A (rilanciato a 10–20 MeV con
`stress_anisotropia.py 3 A`).

In [ ]:
import stress_anisotropia

path = OUT / "stress_anisotropia_results.pkl"
if path.exists():
    with open(path, "rb") as fh:
        aniso_results = pickle.load(fh)
    print("casi nel file:", aniso_results["cases"])
    stress_anisotropia.print_table(aniso_results["results"], aniso_results["cases"])
else:
    print(f"{path} mancante: rigenerarlo con scripts/stress_anisotropia.py")

**Lettura, Caso A (c).** Il bias aggiunto dall'anisotropia è +0.003–0.005 MeV, cioè 0.01 / 0.04 /
0.12 std posteriori a $N$ = 10 / 100 / 1000, sotto la soglia di 0.2. Pull e coverage restano
nominali: a $N = 1000$ il pull medio passa da −0.00 a +0.11, con larghezza 0.97 in entrambi i casi.

Il segno è quello atteso. Con più protoni di alta energia, il modello isotropo legge un $E_n$ un po'
più alto. Il bias in std cresce come $\sqrt N$, perché il bias assoluto è costante mentre la std
scende: oltre $N \approx 1000$ andrebbe ricontrollato (b).

**Casi B e C.** Il run precedente con tutti e tre i casi, riportato nel resoconto (§6.5), dava:

* nessun effetto su $\Omega_n$, come previsto dalla simmetria;
* sul Caso C, un bias aggiunto su $\mu_E$ di 0.02 / 0.04 / 0.07 / 0.15 std a $N$ = 50 / 150 /
  300 / 1000, e ≤ 0.02 std su $\log\sigma_E$.

Quei numeri non sono nel file attuale, che è stato sovrascritto dal rilancio del solo Caso A
(c, dal resoconto, non riprodotto qui). La calibrazione mancata del Caso C (sezione 5) è identica con
le due generazioni, quindi non viene dall'anisotropia.

**Decisione.** La ricostruzione resta isotropa. L'anisotropia è disponibile nel generatore
(`kinematics.sample_cm_angle_anisotropic`) per gli stress test.

## 3. Prior fisico su $\theta_p$ nel Caso A

Nel Caso A, $\theta_p$ è un nuisance per evento, marginalizzato in `forward_model.loglik` con un
prior su $[0, \pi/2]$. Fino alla v0.18 il prior era piatto. Le checklist generavano
$\theta_p \sim U(0,\pi/2)$, coerente con quel prior, e la calibrazione risultava perfetta.

Ma la cinematica fisica non dà $\theta_p$ uniforme. Con scattering isotropo in CM,
$E_p \sim U(0, E_n)$; nel limite classico $\cos^2\theta_p = E_p/E_n$ è quindi uniforme, e (a)

$$p(\theta_p) = \left|\frac{d\cos^2\theta_p}{d\theta_p}\right| = 2\sin\theta_p\cos\theta_p = \sin 2\theta_p .$$

Il prior piatto dà troppo peso agli angoli vicini a 0 e a 90°. Il difetto c'era a ogni energia, ma
emerge solo quando gli eventi vengono dalla cinematica in CM, come nella v0.19.

In [ ]:
theta_grid = grids.theta_p_grid()
theta_p_cm, _ = cm_events(rng, 15.0, 200_000)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(theta_p_cm, bins=45, range=(0, np.pi / 2), density=True, histtype="step", color="k",
        label="cinematica in CM, 15 MeV")
ax.plot(theta_grid, np.sin(2 * theta_grid), label=r"$\sin 2\theta_p$ (prior fisico)")
ax.plot(theta_grid, np.full_like(theta_grid, 2 / np.pi), label="piatto (fino alla v0.18)")
ax.set_xlabel(r"$\theta_p$ [rad]")
ax.set_ylabel("densità")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Effetto sul posterior combinato

Si combinano $N = 1000$ eventi a $E_n = 15$ MeV, generati dalla cinematica in CM e misurati con
$\sigma_{E_p}$ e $\sigma_\theta$, una volta col prior piatto e una volta con quello fisico. Bastano
poche ripetizioni: ognuna costa circa un secondo per prior.

In [ ]:
En_grid = grids.energy_grid()
log_prior_En = np.log(priors.energy_prior(En_grid))
log_flat = np.full(theta_grid.size, -np.log(theta_grid.size))
n_events, n_rep = 1000, 8

residuals = {"piatto": [], "sin 2θ": []}
stds = {"piatto": [], "sin 2θ": []}
for _ in range(n_rep):
    theta_true, Ep_true = cm_events(rng, En_true, n_events)
    D = forward_model.measure(Ep_true, theta_true, SIGMA_EP, SIGMA_THETA, rng)
    for name, lp_theta in (("piatto", log_flat), ("sin 2θ", None)):
        per_event = posterior_A.single_event_posterior(D, En_grid, np.ones_like(En_grid), lp_theta)
        mean, std = posterior_summary(combine.combine_loglik(per_event, log_prior_En), En_grid)
        residuals[name].append(mean - En_true)
        stds[name].append(std)

for name in residuals:
    r, s = np.array(residuals[name]), np.array(stds[name])
    print(f"prior {name:7s}: bias {r.mean():+.3f} +- {r.std(ddof=1) / np.sqrt(n_rep):.3f} MeV,"
          f" std post. {s.mean():.3f} MeV, pull medio {np.mean(r / s):+.2f}")

**Lettura (c, $M = 8$ ripetizioni).** Col prior piatto $E_n$ esce più basso del vero di molte
std posteriori. Col prior fisico il bias è compatibile con zero. La run più grande del resoconto
(20 ripetizioni) dà −0.101 ± 0.009 MeV, cioè pull medio −3.9, contro −0.006 ± 0.008 MeV (c).

Il bias non si riduce con $N$: è comune a tutti gli eventi, come lo scarto relativistico della
sezione 1. Dalla v0.19 il prior di default su $\theta_p$ è `priors.log_recoil_angle_prior`, con pesi
pari alla massa esatta di $\sin 2\theta_p$ in ogni cella della griglia.

Due note:

* il prior è non relativistico; la correzione dipende da $E_n$ (~1% a 20 MeV) e non si
  fattorizza in un prior su $\theta_p$ indipendente da $E_n$ (b);
* il caso di riferimento R1 è definito con il prior piatto, e il suo test lo passa
  esplicitamente. Col prior fisico la σ sarebbe 0.2217 MeV invece di 0.2197, fuori tolleranza.

## 4. `loglik` del Caso A sotto 1 ms

`forward_model.loglik` calcola per ogni evento una tabella $(n_{E_n}, n_\theta) = (500, 400)$ e la
marginalizza su $\theta_p$ con un log-sum-exp. Con la cinematica relativistica il tempo per un evento
era salito sopra il target di 1 ms. Il profilo mostrava due cause:

* circa un terzo del tempo erano page fault su tabelle nuove allocate a ogni passo (ognuna
  occupa 0.8 MB in float32);
* numpy faceva una copia difensiva quando input e output erano viste diverse della stessa memoria.

La versione attuale usa un solo buffer, riusato da tutti i blocchi di eventi, e tutte le operazioni
lavorano in place:

* il residuo è scalato per $\sqrt{0.5}/\sigma_{E_p}$, così il suo quadrato è già il termine
  gaussiano cambiato di segno;
* il termine in $\theta$ si somma una volta per evento, perché non dipende da $E_n$;
* il log-sum-exp su $\theta$ è fatto a mano: `scipy.special.logsumexp` ne costava il 75%;
* `kinematics.proton_energy` divide in place sulla tabella del denominatore.

Il risultato è uguale entro $4\cdot 10^{-6}$ relativo, cioè la precisione di float32 (c). La cella
sotto misura i tempi su questa macchina; i valori di riferimento, misurati su un core
(`taskset -c 0`), sono 0.67 ms per un evento e 0.71 s per 1000 eventi (c).

In [ ]:
log_prior_theta = priors.log_recoil_angle_prior(theta_grid)


def time_loglik(n_events: int, repeats: int) -> float:
    """Tempo minimo di forward_model.loglik su n_events eventi a 15 MeV.

    Args:
        n_events: numero di eventi.
        repeats: ripetizioni; si tiene il minimo, meno sensibile al carico della macchina.

    Ritorna:
        tempo in s per chiamata.
    """
    theta_true, Ep_true = cm_events(rng, En_true, n_events)
    D = forward_model.measure(Ep_true, theta_true, SIGMA_EP, SIGMA_THETA, rng)
    forward_model.loglik(D, En_grid, theta_grid, SIGMA_EP, SIGMA_THETA, log_prior_theta)  # riscaldamento
    best = np.inf
    for _ in range(repeats):
        t0 = time.perf_counter()
        forward_model.loglik(D, En_grid, theta_grid, SIGMA_EP, SIGMA_THETA, log_prior_theta)
        best = min(best, time.perf_counter() - t0)
    return best


print(f"1 evento:     {1e3 * time_loglik(1, 200):.2f} ms   (target < 1 ms)")
print(f"1000 eventi:  {time_loglik(1000, 3):.2f} s    (target < 2 s)")

**Lettura.** I tempi dipendono dalla macchina e dal carico. Il riferimento sono le soglie di
`tests/test_performance.py`, che restano il controllo di regressione. Il target per 1000 eventi era
stato ridefinito a 2 s già prima: il limite è il calcolo numpy denso su $10^3 \times 500 \times 400$
celle, non il codice Python.

## 5. Punto aperto: calibrazione del Caso C a 10–20 MeV

A 0.5–6 MeV il Caso C era calibrato. A 10–20 MeV la checklist dà (c, notebook 05):

* $\Omega_n$ calibrata;
* $\mu_E$ con larghezza del pull 1.24–1.30 a $N \ge 300$;
* $\log\sigma_E$ con pull medio crescente con $N$, da +0.10 a +0.61.

### Una causa candidata: lo spettro esce dal dominio

La verità è generata con $\mu_E \sim U(13, 17)$ MeV e $\sigma_E$ log-uniforme in $(0.8, 2.4)$ MeV,
senza troncatura. La ricostruzione integra però $E_n$ solo su $[10, 20]$ MeV. La frazione dello
spettro vero fuori dal dominio è

$$p_\mathrm{out} = \Phi\!\left(\frac{10-\mu_E}{\sigma_E}\right) + 1 - \Phi\!\left(\frac{20-\mu_E}{\sigma_E}\right).$$

Nel caso peggiore, $\mu_E = 13$ e $\sigma_E = 2.4$, si ha $\Phi(-1.25) + 1 - \Phi(2.92) = 0.106 + 0.002
\approx 0.107$: un evento su dieci ha un'energia che il modello non può spiegare. Nel vecchio dominio
il margine c'era: $\mu_E \in [2.5, 4]$ e $\sigma_E \le 0.6$ danno $\mu_E \pm 3\sigma_E \subset
[0.7, 5.8] \subset [0.5, 6]$. Il passaggio a 10–20 MeV ha conservato $\sigma_E/\mu_E$ ma non il
margine di $3\sigma_E$: $13 - 3 \cdot 2.4 = 5.8 < 10$.

In [ ]:
# intervalli dei generatori della checklist del Caso C, prima (0.5-6 MeV) e dopo (10-20 MeV)
GENERATORS = {"0.5–6 MeV": ((2.5, 4.0), (0.2, 0.6), (0.5, 6.0)),
              "10–20 MeV": ((13.0, 17.0), (0.8, 2.4), (EN_MIN, EN_MAX))}


def out_of_domain(mu: np.ndarray, sigma: np.ndarray, lo: float, hi: float) -> np.ndarray:
    """Frazione di uno spettro gaussiano fuori dal dominio di ricostruzione [lo, hi].

    Args:
        mu: media dello spettro, MeV, forma qualsiasi.
        sigma: deviazione standard, MeV, forma compatibile.
        lo, hi: estremi del dominio in E_n, MeV.

    Ritorna:
        p_out, adimensionale in [0, 1], forma del broadcasting.
    """
    return norm.cdf((lo - mu) / sigma) + norm.sf((hi - mu) / sigma)


fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, (name, (mu_range, sigma_range, (lo, hi))) in zip(axes, GENERATORS.items()):
    mu_g = np.linspace(*mu_range, 200)
    sigma_g = np.geomspace(*sigma_range, 200)
    p = out_of_domain(mu_g[None, :], sigma_g[:, None], lo, hi)
    im = ax.pcolormesh(mu_g, sigma_g, np.log10(np.clip(p, 1e-12, None)), vmin=-6, vmax=-1, shading="auto")
    ax.contour(mu_g, sigma_g, p, levels=[0.01], colors="w")
    ax.set_xlabel(r"$\mu_E$ [MeV]")
    ax.set_ylabel(r"$\sigma_E$ [MeV]")
    ax.set_title(f"{name}: max $p_{{out}}$ = {p.max():.3f}")
fig.colorbar(im, ax=axes, label=r"$\log_{10} p_\mathrm{out}$ (curva bianca: 1%)")
plt.show()

# frazione di esperimenti della checklist con p_out >= 1%, estraendo (mu, sigma) come il generatore
mu_s = rng.uniform(13.0, 17.0, 200_000)
sigma_s = np.exp(rng.uniform(np.log(0.8), np.log(2.4), 200_000))
print(f"10–20 MeV: frazione di esperimenti con p_out >= 1%: {np.mean(out_of_domain(mu_s, sigma_s, EN_MIN, EN_MAX) >= 0.01):.2f}")

**Lettura.** Nel vecchio dominio $p_\mathrm{out}$ era trascurabile ovunque. Nel nuovo supera l'1%
in circa un terzo degli esperimenti della checklist (a, è un integrale gaussiano).

Il notebook 05 (sezione 6) divide gli esperimenti della checklist in base a questa soglia (c):

* con $p_\mathrm{out} \ge 1\%$, il pull medio di $\log\sigma_E$ è +0.55 / +0.79 / +0.84 / +1.36 a
  $N$ = 50 / 150 / 300 / 1000;
* con $p_\mathrm{out} < 1\%$, è −0.08 / −0.07 / +0.12 / +0.20, e la larghezza del pull di $\mu_E$
  è 0.96 / 0.79 / 1.18 / 1.00;
* nel primo gruppo il pull di $\mu_E$ è correlato con $\mu_\mathrm{vero} - 15$ (0.68–0.76): gli
  spettri spostati verso un bordo vengono tirati verso il centro.

La divisione è un fatto Monte Carlo (c). Che la troncatura sia *la* causa è ancora un'ipotesi
(d): va confermata cambiando una cosa sola e rifacendo la checklist.

### Correzioni possibili (da decidere, non applicate)

1. **Restringere il generatore**: $\sigma_E \le 1$ MeV con $\mu_E \in [13, 17]$ riporta il margine
   di $3\sigma_E$ dentro $[10, 20]$. È il test più pulito dell'ipotesi, ma il rapporto
   $\sigma_E/\mu_E$ non è più quello della v0.18.
2. **Allargare la griglia di $E_n$ del Caso C** (per esempio fino a `EN_MAX_WIDE`), tenendo il
   generatore com'è. Il modello potrebbe spiegare tutto lo spettro, ma la griglia diventa più cara
   e il prior su $E_n$ cambia significato.

Entrambe richiedono di rifare la checklist del Caso C, un run pesante che va lanciato con un tetto
di memoria. Gli stessi intervalli di generazione sono negli script di stress del Caso C
(`stress_sorgente`, `stress_anisotropia`, `stress_spettro`, `robustezza_prior`,
`correlazione_stadi`, `modo_log_sigma`), che andranno allineati alla scelta.

## Riassunto

| Modifica | Perché | Esito | |
|---|---|---|---|
| Cinematica relativistica esatta | lo scarto classico (fino a 0.5 $\sigma_{E_p}$) è comune: bias −0.06 MeV a 15 MeV, plateau per $N \gtrsim 500$ | applicata; nessun plateau nella checklist A | (a), (c) |
| Isotropia in CM contro ENDF/B-VIII.0 | $f(-1)/f(0)$ fino a 1.12 a 20 MeV | bias aggiunto ≤ 0.12 std (A, $N = 1000$), ≤ 0.15 std ($\mu_E$, C); ricostruzione isotropa | (c) |
| Prior $\sin 2\theta_p$ nel Caso A | il prior piatto è incoerente con la cinematica in CM | bias combinato da −0.10 a −0.006 MeV a $N = 1000$ | (c) |
| `loglik` con buffer unico in place | page fault e copie difensive | 0.67 ms per evento, 0.71 s per 1000 | (c) |
| Caso C a 10–20 MeV | $\mu_E$ e $\log\sigma_E$ non calibrati | candidato: spettro vero fuori da [10, 20] (fino al 10.7%); correzione da decidere | (c), (d) |